# 03 — Phase 2: sub-pixel shoreline extraction

Thin Colab driver for `src/extract.py` (heavy logic lives in the module). Follows
the verification order in **PHASE2_SPEC.md §5**, anchored on the clean 2023
Sentinel-2 single scene (season 2022-2023) for the fetch → classify → extract →
visual-QC steps.

**Locked design (D1–D7):** scene-level extraction (one tide per scene); Series A
(38 annual products) + Series B (dense all-season 1999–2025); a LOCAL
sand/water/whitewater/other MLP per sensor group (TM/OLI/MSI); the water index is
a parameter (default MNDWI) with the Otsu/peaks threshold taken on **sand ∪ water
pixels only**; marching-squares sub-pixel contour; tidal-channel closure lines.

**Operator artifacts required before the classifier/extraction steps** (digitise
in QGIS, upload via the GitHub web interface — see the final cell):
`data/shoreline_search_zone.geojson`, `data/training_polygons.geojson`,
`data/reference_shorelines/`.


## Setup — clone-or-pull, authenticate GEE, import & reload `src/`

In [ ]:
import os
REPO_DIR = '/content/Shoreline-Dynamics'
# Clone-or-pull so Colab always runs the latest committed code.
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone https://github.com/SKPrince1911/Shoreline-Dynamics.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='shoreline-analysis')
print('GEE initialized:', ee.String('ok').getInfo())

In [ ]:
import sys; sys.path.append('.')
import importlib
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from src import config, data, extract
# Pick up freshly pulled modules without a kernel restart (config before its
# dependents). extract is on the reload list so edits to src/extract.py land.
importlib.reload(config); importlib.reload(data); importlib.reload(extract)

aoi = ee.Geometry.Polygon(config.aoi_coordinates())
print('feature dim:', extract.feats_dim(),
      '| default index:', config.WATER_INDEX_DEFAULT,
      '| threshold:', config.THRESHOLD_METHOD_DEFAULT)
print('sensor groups:', extract.SENSOR_GROUP)

## 1. Scene lists — Series A (annual) and Series B (dense)

Series A explodes the approved inventory into one row per **acquisition date**
(same-overpass granules mosaicked; multi-date composites split), so every scene
keeps its own timestamp for Phase 3.

In [ ]:
annual = extract.build_scene_list_annual('outputs/image_inventory.csv')
n_years = annual['dry_year'].nunique()
print(f'Series A: {len(annual)} scenes across {n_years} dry-season-years')
assert n_years == 38, f'expected 38 non-gap years, got {n_years}'
assert annual['acq_datetime_utc'].notna().all(), 'found a NaT timestamp!'
print('sensors:', annual['sensor'].value_counts().to_dict())
# Composite temporal spread (E_temporal precursor): years whose contributing
# acquisitions span > 60 days mix seasons within one annual shoreline.
spread = annual.groupby('dry_year')['composite_date_spread_days'].first()
flagged = spread[spread > config.COMPOSITE_SPREAD_FLAG_DAYS].to_dict()
print(f'composite spread > {int(config.COMPOSITE_SPREAD_FLAG_DAYS)} d (E_temporal flag):', flagged)
annual.head(12)

Series B is a **large, all-season GEE query**, run **one calendar year at
a time** (chunked ≤8 aggregations per request to dodge HTTP 429, and to stay
under the per-request memory limit). It prints a per-year progress line and can
take **tens of minutes to ~2 hours**; occasional `503` warnings are transient
and auto-retried, so let it run. The envelope runs to **2026-04-30** so the full
2025-2026 dry season is captured; the boundary dry_years (1999, 2026) are
all-season-incomplete and carry `season_complete=False` so they can't slip into a
per-year seasonal statistic. It writes `outputs/scene_list_dense.csv` and is not
needed for the single-scene QC below.

In [ ]:
# verbose=True (default) prints "YYYY: k kept of n | running total ..." per year.
dense = extract.build_scene_list_dense()   # 1999-01-01 .. 2026-04-30, L7/L8/L9/S2
print(f'Series B: {len(dense)} scenes kept '
      f'(cloud <= {config.DENSE_CLOUD_MAX_PCT}%, coverage >= {config.DENSE_COVERAGE_MIN_PCT}%)')
print('per sensor:', dense['sensor'].value_counts().to_dict())
print('SLC-off L7 scenes:', int(dense['slc_off'].sum()))
# Boundary dry_years whose all-season window (Nov(Y-1)-Oct(Y)) is only partially
# inside the query envelope -> EXCLUDE from any per-dry_year seasonal statistic.
incomplete = sorted(dense.loc[~dense['season_complete'], 'dry_year'].unique().tolist())
print('season_complete=False dry_years:', incomplete)
# Effective sampling density per dry_year: 'clean' scenes give a full transect
# crossing; SLC-off L7 scenes have ~22% striping gaps, so a transect sitting in a
# gap contributes no crossing that date — what matters for whether the
# Lomb-Scargle slope estimate converges (raw scene counts overstate density).
bk = dense.assign(_clean=~dense['slc_off'].astype(bool)).groupby('dry_year').agg(
    n=('image_id', 'size'), slc_off=('slc_off', 'sum'), clean=('_clean', 'sum'),
    complete=('season_complete', 'first'))
bk

## 2. Fetch scenes — visual QC (RGB + valid mask)

Fetch is tiled onto a fixed **EPSG:32646** grid (pixel-aligned across scenes).

Two things about this QC: **(a)** the ~6 km demo window over Kolatoli / Marine
Drive is a small test patch used **only until `shoreline_search_zone.geojson`
exists** (it keeps 10 m Sentinel-2 within Colab RAM); **(b)** an *individual
Landsat* scene images only **part** of the AOI, because the coast straddles
WRS-2 **paths 135 (south) and 136 (north)** — this demo window is northern, so
only path-136 scenes cover it. That partial coverage is **expected**, and is
exactly why extraction is per-scene with vector merging (D1). The QC below picks
a scene whose footprint actually intersects the window rather than a blind
`.iloc[0]`.

In [ ]:
from shapely.geometry import box
demo_ll = box(92.00, 21.33, 92.06, 21.42)     # (minlon, minlat, maxlon, maxlat)
demo_utm = extract._to_utm(demo_ll)

def rgb_from_scene(sc, pmin=2, pmax=98):
    # Per-scene 2-98% contrast stretch on the VALID pixels (a fixed 0..0.3
    # stretch looks flat/odd on turbid GBM water). Invalid pixels -> black, not
    # matplotlib's NaN artifacts.
    rgb = np.dstack([sc.bands['red'], sc.bands['green'], sc.bands['blue']]).astype('float32')
    fin = np.isfinite(rgb).all(axis=2)
    if fin.any():
        lo, hi = np.nanpercentile(rgb[fin], [pmin, pmax])
        if not (hi > lo):
            lo, hi = 0.0, 0.3
    else:
        lo, hi = 0.0, 0.3
    return np.nan_to_num(np.clip((rgb - lo) / (hi - lo), 0, 1), nan=0.0)

def show_scene(sc, title=''):
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    ax[0].imshow(rgb_from_scene(sc)); ax[0].set_title(title + ' — true colour (2-98% stretch)')
    ax[0].axis('off')
    ax[1].imshow(sc.valid.astype(float), cmap='gray', vmin=0, vmax=1)  # fixed 0..1 so all-valid = white
    ax[1].set_title('valid mask (white = usable)'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
    validpct = 100 * sc.valid.mean()
    print(title, sc.sensor, sc.image_id[:44], '| acq', sc.acq_datetime_utc,
          '| georef', sc.georef_rmse_m, 'm | valid %.1f%%' % validpct)
    # Per-band diagnostics reveal a degenerate scene at a glance (e.g. one band
    # flat, or values in the thousands = wrong scaling / band mapping).
    for b in ('red', 'green', 'blue', 'swir1'):
        a = sc.bands[b]
        print('   %-6s min=%.3f  median=%.3f  max=%.3f'
              % (b, float(np.nanmin(a)), float(np.nanmedian(a)), float(np.nanmax(a))))
    if validpct < 40:
        print('   WARNING: low valid coverage — the scene is mostly cloud/nodata '
              'over this window; the true-colour panel will be mostly black.')
    gmin, gmax = float(np.nanmin(sc.bands['green'])), float(np.nanmax(sc.bands['green']))
    assert -0.5 <= gmin and gmax <= 3.0, (
        'reflectance far outside the SR range — likely a band-mapping or scaling '
        'error (green min=%.2f max=%.2f)' % (gmin, gmax))

def row_for(df, sensor=None, dry_year=None):
    q = df
    if sensor is not None: q = q[q['sensor'] == sensor]
    if dry_year is not None: q = q[q['dry_year'] == dry_year]
    return q.iloc[0]

def first_covering(df, window_ll, sensor=None, dry_year=None):
    # First scene whose granule footprint actually intersects window_ll (EPSG:4326).
    # A single Landsat scene images only part of the AOI (WRS-2 paths 135 & 136),
    # so a plain .iloc[0] can return a scene that never saw the demo window.
    q = df
    if sensor is not None: q = q[q['sensor'] == sensor]
    if dry_year is not None: q = q[q['dry_year'] == dry_year]
    minx, miny, maxx, maxy = window_ll.bounds
    win = ee.Geometry.Rectangle([minx, miny, maxx, maxy])
    for _, r in q.iterrows():
        gid = str(r['image_id']).split(',')[0]
        if ee.Image(extract._ee_asset_id(gid, r['sensor'])).geometry().intersects(win, 1).getInfo():
            return r
    print('no', sensor or '', 'scene intersects the demo window (checked', len(q),
          'candidates) — skipping')
    return None

In [ ]:
# The 2023 (2022-2023) clean single Sentinel-2 product (confirmed to cover the window).
s2_row = first_covering(annual, demo_ll, sensor='S2', dry_year=2023)
sc_s2 = extract.fetch_scene(s2_row, region_utm=demo_utm) if s2_row is not None else None
if sc_s2 is not None:
    show_scene(sc_s2, '2023 (2022-2023)')

In [ ]:
# One Landsat-5 and one Landsat-8 scene that ACTUALLY cover the demo window
# (a blind iloc[0] can be a path-135 scene that never imaged this northern patch).
for _sensor, _title in [('L5', 'Landsat-5 sample'), ('L8', 'Landsat-8 sample')]:
    _row = first_covering(annual, demo_ll, sensor=_sensor)
    if _row is not None:
        show_scene(extract.fetch_scene(_row, region_utm=demo_utm), _title)

## 2b. Export a scene as a GeoTIFF for QGIS labelling

Write the 2023 Sentinel-2 scene to a georeferenced multi-band GeoTIFF
(EPSG:32646, R/G/B/SWIR1) on the exact grid the pipeline extracts from — so the
`training_polygons.geojson` and `reference_shorelines/` you digitise over it line
up pixel-for-pixel with what the classifier and contour see. Open the `.tif` in
QGIS (it carries its CRS and band names; SWIR1 makes the wet/dry line obvious).

Before the search zone exists this exports the small demo window; once
`shoreline_search_zone.geojson` is uploaded, re-run to export the full coastal
ribbon (and repeat for a Landsat scene per sensor group you want to label).

In [ ]:
export_dir = ('/content/drive/MyDrive/shoreline_dynamics'
              if os.path.isdir('/content/drive/MyDrive') else 'outputs')
os.makedirs(export_dir, exist_ok=True)
tif = os.path.join(export_dir, 'scene_2023_S2_utm46n.tif')
extract.export_scene_geotiff(
    s2_row, tif, bands=('red', 'green', 'blue', 'swir1'),
    region_utm=None if os.path.exists(config.SEARCH_ZONE_PATH) else demo_utm,
)
print('wrote', tif, '-> open in QGIS (EPSG:32646) to digitise labels / references')

## 3. Train the Sentinel-2 (MSI) classifier

**Requires `data/training_polygons.geojson`** (class ∈ {other, sand, whitewater,
water}). For a real run also digitise `data/shoreline_search_zone.geojson` first,
so training scenes are fetched over the whole coastal band and every polygon
contributes. **Gate:** per-class precision/recall ≥ 0.90 for *water* and *sand*;
if not, add training polygons and re-run.

In [ ]:
TRAIN_PATH = 'data/training_polygons.geojson'
clf_msi = None
if not os.path.exists(TRAIN_PATH):
    print('BLOCKED:', TRAIN_PATH, 'not found.')
    print('Digitise it in QGIS (see the final cell), upload via the GitHub web')
    print('interface, then `git pull` in the setup cell and re-run.')
else:
    tp = gpd.read_file(TRAIN_PATH)
    print('label polygons by class:', tp['class'].value_counts().to_dict())
    # Fetch over the full search zone if present, else the demo window.
    region = None if os.path.exists(config.SEARCH_ZONE_PATH) else demo_utm
    train_rows = annual[annual['sensor'] == 'S2'].head(5)
    scenes = [extract.fetch_scene(r, region_utm=region) for _, r in train_rows.iterrows()]
    fit_scenes, hold = scenes[:-1], scenes[-1]     # hold out one scene (different date)
    X, y = extract.build_training_set(fit_scenes, TRAIN_PATH)
    uniq, cnt = np.unique(y, return_counts=True)
    print('training samples:', X.shape, '| per class:', dict(zip(uniq.tolist(), cnt.tolist())))
    clf_msi = extract.train_classifier(X, y, 'MSI')
    Xh, yh = extract.build_training_set([hold], TRAIN_PATH)
    print(extract.evaluate_classifier(clf_msi, Xh, yh, write_report=True, sensor_group='MSI'))

## 4. Extract the 2023 shoreline + visual QC

Classify → MNDWI → Otsu on the **sand ∪ water interface** → marching-squares
sub-pixel contour → clip to the search zone and follow the tidal-channel closure
lines. Visually: the red contour should track the wet/dry land–water boundary,
sit off the Marine Drive road edge, not run up the tidal channels (cyan), and
not hug the cloud-mask boundary.

In [ ]:
if not os.path.exists(TRAIN_PATH):
    print('BLOCKED: train the MSI classifier first (needs data/training_polygons.geojson).')
    line = None
else:
    settings = extract.default_settings(water_index_name='mndwi', threshold_method='otsu')
    # Full extraction region if the search zone exists, else the demo window.
    if os.path.exists(config.SEARCH_ZONE_PATH):
        sc = extract.fetch_scene(s2_row)
    else:
        sc = sc_s2
        print('note: no search zone yet -> extracting within the demo window only.')
    clf = clf_msi if clf_msi is not None else extract.load_classifier('MSI')
    labels = extract.classify_scene(sc, clf)
    idx = extract.water_index(sc, 'mndwi')
    thr = extract.interface_threshold(idx, labels, 'otsu')
    contours = extract.extract_contour(idx, thr, sc.valid, sc.transform)
    line = extract.filter_contours(contours, settings['search_zone'], settings['channel_lines'])
    print('interface Otsu threshold (sand-union-water only):', round(float(thr), 4))
    print('raw contour segments:', len(contours),
          '| final shoreline length_m:', None if line is None else round(line.length, 1))

    t, (H, W) = sc.transform, sc.valid.shape
    extent = [t.c, t.c + W * t.a, t.f + H * t.e, t.f]   # [xmin, xmax, ymin, ymax] UTM
    overlay = np.zeros((H, W, 3))
    overlay[labels == config.CLASS_SAND] = [0.96, 0.88, 0.30]
    overlay[labels == config.CLASS_WATER] = [0.17, 0.50, 0.72]
    overlay[labels == config.CLASS_WHITEWATER] = [1.0, 1.0, 1.0]
    fig, ax = plt.subplots(1, 3, figsize=(17, 6))
    ax[0].imshow(rgb_from_scene(sc)); ax[0].set_title('RGB'); ax[0].axis('off')
    ax[1].imshow(overlay); ax[1].set_title('classified sand / water / whitewater'); ax[1].axis('off')
    ax[2].imshow(rgb_from_scene(sc), extent=extent, origin='upper')
    if line is not None:
        for ln in extract._as_line_list(line):
            xs, ys = ln.xy; ax[2].plot(xs, ys, '-', color='red', lw=1.2)
    for cl in settings['channel_lines']:
        xs, ys = cl.xy; ax[2].plot(xs, ys, color='cyan', lw=2)
    ax[2].set_aspect('equal'); ax[2].set_title('sub-pixel shoreline (red) + channels (cyan)')
    plt.tight_layout(); plt.show()

## 5. Zoomed comparison — how tightly the contour follows the boundary

In [ ]:
if 'line' in globals() and line is not None:
    mid = line.interpolate(0.5, normalized=True)
    half = 750  # metres
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb_from_scene(sc), extent=extent, origin='upper')
    for ln in extract._as_line_list(line):
        xs, ys = ln.xy; ax.plot(xs, ys, '-', color='red', lw=1.6)
    ax.set_xlim(mid.x - half, mid.x + half); ax.set_ylim(mid.y - half, mid.y + half)
    ax.set_aspect('equal'); ax.set_title('zoom: sub-pixel contour vs land/water boundary')
    plt.show()
else:
    print('Run the extraction cell first (needs the trained classifier).')

## 6. Batch extraction — Series A, then Series B

Run only once the single-scene QC looks right **and** a classifier exists for
every sensor group the list needs (Series A spans TM/OLI/MSI). `extract_all`
checkpoints to `outputs/shorelines/sds_scenes.geojson` after **every** scene and
resumes from it, so a Colab disconnect never costs the whole run.

In [ ]:
def _missing_clfs(groups):
    """Sensor groups with no models/clf_<group>_<version>.joblib on disk."""
    return [g for g in groups if not os.path.exists(os.path.join(
        extract.MODELS_DIR, f'clf_{g}_{extract.DEFAULT_CLASSIFIER_VERSION}.joblib'))]

# Series A needs a classifier for every sensor group its scenes use (TM/OLI/MSI).
needed_A = sorted({extract.sensor_group(s) for s in annual['sensor'].unique()})
missing_A = _missing_clfs(needed_A)
if missing_A:
    print('BLOCKED: Series A needs classifiers for', needed_A,
          '- still to train:', missing_A)
    print('Train each missing sensor group (see the training cell) before batch extraction.')
else:
    settings = extract.default_settings()
    gdfA = extract.extract_all(annual, settings)         # Series A (~60 scenes)
    merged = extract.merge_annual(gdfA)                  # per-segment source for Phase 3
    os.makedirs(extract.SHORELINE_DIR, exist_ok=True)
    merged.to_file(extract.SDS_ANNUAL_PATH, driver='GeoJSON')
    print('Series A scenes extracted:', len(gdfA), '-> merged segments:', len(merged))

# Series B (hundreds of scenes) needs TM+OLI+MSI. When ready:
#   missing_B = _missing_clfs(['TM', 'OLI', 'MSI'])
#   if not missing_B:
#       gdfB = extract.extract_all(dense, settings)
# Push outputs/ to Drive + GitHub with the save_outputs() cell from notebook 02.

## 7. Transects + benchmark / inter-sensor bias (deliverables)

Transects (DSAS/QSCAT convention) are cast shore-normal from a digitised
baseline; they are what `benchmark_extraction()` and `intersensor_bias()` measure
along. Requires `data/baseline.geojson` (see the final cell).

In [ ]:
if os.path.exists(extract.BASELINE_PATH):
    transects = extract.build_transects()   # -> outputs/transects.geojson (EPSG:4326)
    print(f'built {len(transects)} transects @ {int(extract.TRANSECT_SPACING_M)} m spacing, '
          f'{int(extract.TRANSECT_LENGTH_M)} m long -> {extract.TRANSECTS_PATH}')
else:
    print('BLOCKED:', extract.BASELINE_PATH, '- digitise the baseline (see the final cell).')

# Once transects + the 10 reference shorelines exist, the deliverables run as:
#   refs = gpd.read_file('data/reference_shorelines/<file>.geojson')  # image_id per line
#   ref_scenes = [extract.fetch_scene(r) for _, r in <reference rows>.iterrows()]
#   extract.benchmark_extraction(ref_scenes, refs)   # mndwi/ndwi/aweinsh x otsu/weighted_peaks
#   extract.intersensor_bias(gdfA)                   # L5/L7 2001/2005, L8/L9 x S2 overlaps

## Operator artifacts to digitise in QGIS (upload via the GitHub web interface)

These are the manual inputs the extraction reads. Upload them under `data/` on
GitHub, then `git pull` in the setup cell.

**1. `data/shoreline_search_zone.geojson`** — one polygon (EPSG:4326) enclosing
every plausible shoreline position 1988–2025: roughly ±300 m about the present
shoreline along the open Marine Drive coast, widening to ±800–1000 m within ~3 km
of the Bakkhali and Naf river mouths. Replaces CoastSat's scalar `max_dist_ref`
and keeps extraction human-controlled and auditable.

**2. `data/training_polygons.geojson`** — labelled polygons with a `class` field
in {`other`, `sand`, `whitewater`, `water`}. Digitise ~6 scenes per sensor group
(TM = L4/L5/L7, OLI = L8/L9, MSI = S2), spread across dry/monsoon and along the
AOI (Bakkhali mouth, Kolatoli/Marine Drive, Himchari cliffs, Inani, Teknaf, Shah
Porir Dwip). Aim for ≥5,000 labelled pixels per class per sensor group.

**3. `data/reference_shorelines/`** — 10 manually digitised validation
shorelines (LineStrings, EPSG:4326) with an `image_id` property matching the
scene, digitised at native resolution (~1:2,000) along the wet/dry line. Used by
`benchmark_extraction()` to pick the operational index/threshold and by
`intersensor_bias()`.

**4. `data/baseline.geojson`** — a single continuous LineString digitised
**landward of and roughly parallel to** the whole coast (from the north AOI edge
to Shah Porir Dwip, ~200–500 m inland of the present shoreline). Keep it smooth —
it sets each transect's direction, so avoid kinks and self-intersections; digitise
it seaward-consistently (the code casts normals westward, toward the Bay).
`build_transects()` casts shore-normal transects from it every
`TRANSECT_SPACING_M` (50 m), `TRANSECT_LENGTH_M` (1500 m) long, with stable
`transect_id`s, to `outputs/transects.geojson`.

Generated outputs (`outputs/scene_list_dense.csv`, `outputs/transects.geojson`,
`outputs/shorelines/*.geojson`, `outputs/extraction_log.csv`, `models/*.joblib`)
are pushed to Drive + GitHub by the existing `save_outputs()` — no manual upload.
